In [ ]:
#An analysis of relevant stock price responses to highly publicized events.
#Focused on the Stargate data center in Abilene, Texas, developed by Crusoe.

#Requires installation of the following, if necessary:
#pip install os yfinance numpy pandas matplotlib seaborn scipy statsmodels

In [86]:
#necessary imports
import os
import requests
import yfinance as yf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sb
from scipy import stats as st
from statsmodels.regression.linear_model import OLS
from datetime import datetime, timedelta

#select tickers
equipmentTickers = [
    'ETN',
    'GEV',
    'PH'
]

powerTickers = [
    'VST',
    'CEG'
]

smrTickers = [
    'SMR',
    'OKLO',
    'NNE'
]

controlTickers = [
    'SPY'
]

allTickers = equipmentTickers + powerTickers + smrTickers + controlTickers

#events
events = [
    ("2024-06-05", "Crusoe begins construction"),
    ("2024-10-01", "Phase One build-to-suit construction announced"),
    ("2025-01-21", "White House announces Stargate at press conference"),
    ("2025-03-07", "OpenAI and Oracle announce planned GPU deployment"),
    ("2025-03-10", "Phase Two construction (six buildings) underway"),
    ("2025-05-22", "JPMorgan and Crusoe give substantial funds"),
    ("2025-09-23", "First building goes live"),
    ("2026-03-07", "Expansion canceled; winter outage revealed"),
]

eventsdf = pd.DataFrame(events, columns = ['date', 'eventDescription'])
eventsdf['date'] = pd.to_datetime(eventsdf['date'])
dates = eventsdf['date']

#download daily ticker prices (at close)
#begin one year before the first event on the list
downloadWindowStart = dates.min() - timedelta(days = 367)
downloadWindowEnd = datetime.today().strftime('%Y-%m-%d')

allPrices = yf.download(
    tickers = allTickers,
    start = downloadWindowStart,
    end = downloadWindowEnd,
    auto_adjust = True,
    progress = False
)["Close"]
allPrices.fillna(0, inplace = True)

allReturns = ((allPrices.shift(-1) / allPrices).shift(1)).tail(-1)
allLogReturns = np.log(allReturns)
allReturns.fillna(0, inplace = True)
allLogReturns.fillna(0, inplace = True)
allReturns.replace([np.inf, -np.inf], 0)
allLogReturns.replace([np.inf, -np.inf], 0)

allPricesDestination = "data/allPrices.csv"
allReturnsDestination = "data/allReturns.csv"
allLogReturnsDestination = "data/allLogReturns.csv"

allPrices.to_csv(allPricesDestination)
allReturns.to_csv(allReturnsDestination)
allLogReturns.to_csv(allLogReturnsDestination)

In [87]:
#compute returns for equal-weighted portfolios of each treatment category

ecolumns = []
le = len(equipmentTickers)
for etick in equipmentTickers:
    ecolumns.append(etick)
eReturns = pd.DataFrame(columns = ecolumns)
for ecol in ecolumns:
    eReturns[ecol] = allReturns[ecol]

eReturns.insert(
    loc = len(ecolumns),
    column = 'weightedReturn',
    value = 0.0
)
for etick in equipmentTickers:
    eReturns.insert(
        loc = len(ecolumns),
        column = f'{etick} weight',
        value = 0.0
    )
    
for tradingDay in eReturns.itertuples():
    enz = []
    for etick in equipmentTickers:
        zero = getattr(tradingDay, etick) == 0
        nonzero = getattr(tradingDay, etick) != 0
        if zero: eReturns.loc[tradingDay.Index, f'{etick} weight'] = 0
        if nonzero: enz.append(etick)

    for etick in enz:
        eweight = 1 / (len(enz))
        eReturns.loc[tradingDay.Index, f'{etick} weight'] = eweight
        eReturns.loc[tradingDay.Index, 'weightedReturn'] = (
            eReturns.loc[tradingDay.Index, 'weightedReturn'] +
            (eweight * eReturns.loc[tradingDay.Index, etick])
        )

eReturns.replace([np.inf, -np.inf], 0)
eReturnsDestination = "data/eReturns.csv"
eReturns.to_csv(eReturnsDestination)

                  
pcolumns = []
lp = len(powerTickers)
for ptick in powerTickers:
    pcolumns.append(ptick)
pReturns = pd.DataFrame(columns = pcolumns)
for pcol in pcolumns:
    pReturns[pcol] = allReturns[pcol]

pReturns.insert(
    loc = len(pcolumns),
    column = 'weightedReturn',
    value = 0.0
)
for ptick in powerTickers:
    pReturns.insert(
        loc = len(pcolumns),
        column = f'{ptick} weight',
        value = 0.0
    )
    
for tradingDay in pReturns.itertuples():
    pnz = []
    for ptick in powerTickers:
        zero = getattr(tradingDay, ptick) == 0
        nonzero = getattr(tradingDay, ptick) != 0
        if zero: pReturns.loc[tradingDay.Index, f'{ptick} weight'] = 0
        if nonzero: pnz.append(ptick)

    for ptick in pnz:
        pweight = 1 / (len(pnz))
        pReturns.loc[tradingDay.Index, f'{ptick} weight'] = pweight
        pReturns.loc[tradingDay.Index, 'weightedReturn'] = (
            pReturns.loc[tradingDay.Index, 'weightedReturn'] +
            (pweight * pReturns.loc[tradingDay.Index, ptick])
        )

pReturns.replace([np.inf, -np.inf], 0)
pReturnsDestination = "data/pReturns.csv"
pReturns.to_csv(pReturnsDestination)


scolumns = []
ls = len(smrTickers)
for stick in smrTickers:
    scolumns.append(stick)
sReturns = pd.DataFrame(columns = scolumns)
for scol in scolumns:
    sReturns[scol] = allReturns[scol]

sReturns.insert(
    loc = len(scolumns),
    column = 'weightedReturn',
    value = 0.0
)
for stick in smrTickers:
    sReturns.insert(
        loc = len(scolumns),
        column = f'{stick} weight',
        value = 0.0
    )
    
for tradingDay in sReturns.itertuples():
    snz = []
    for stick in smrTickers:
        zero = getattr(tradingDay, stick) == 0
        nonzero = getattr(tradingDay, stick) != 0
        if zero: sReturns.loc[tradingDay.Index, f'{stick} weight'] = 0
        if nonzero: snz.append(stick)

    for stick in snz:
        sweight = 1 / (len(snz))
        sReturns.loc[tradingDay.Index, f'{stick} weight'] = sweight
        sReturns.loc[tradingDay.Index, 'weightedReturn'] = (
            sReturns.loc[tradingDay.Index, 'weightedReturn'] +
            (sweight * sReturns.loc[tradingDay.Index, stick])
        )

sReturns.replace([np.inf, -np.inf], 0)
sReturnsDestination = "data/sReturns.csv"
sReturns.to_csv(sReturnsDestination)

In [ ]:
#compute factor betas using Fama-French three-factor model
from pandas_datareader.famafrench import FamaFrenchReader as ff
for eventDate, _ in events:
    